# Prompt 1 completion — Colab + Drive

This notebook completes only the Track-B Prompt-1 data and identity barrier. It does not train Track A, start B0–B4, or open the 52-case S cohort. Persistent artifacts and state stay on Drive; temporary inference output stays under `/content`.

First session: run all cells in order with `MAX_CASES=2`. After the GPU smoke succeeds, set `MAX_CASES=None` and run the final RUN/RESUME cell. A later session needs only this configuration cell, the Drive-mount cell, and the final RUN/RESUME cell.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# Single editable configuration cell. These defaults follow the existing iac_runs Drive layout.
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
NNUNET_RESULTS = DRIVE_ROOT / 'nnUNet_results'
TRACKB_CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = 'REPLACE_WITH_STAGE0_COLAB_COMMIT_SHA'  # use the SHA delivered with this notebook
NUM_WORKERS = 2       # runner hard-caps this at 2 for Colab RAM
DEVICE = 'cuda'
MAX_CASES = 2         # 2 for first GPU smoke; None for the full resumable 480-case run
FORCE_REBUILD = False # valid artifacts are never overwritten; use only to replace invalid files
EXPORT_TRUE_SOFTMAX = True
RETRY_FAILED = True

REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
CONFIG_PATH = Path('/content/prompt1_completion_config.json')

def bootstrap_repo():
    if PINNED_COMMIT.startswith('REPLACE_'):
        raise ValueError('Set PINNED_COMMIT to the delivered stage0/colab commit SHA')
    if not REPO.is_dir():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
    head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    wanted = subprocess.check_output(['git', 'rev-parse', PINNED_COMMIT], cwd=REPO, text=True).strip()
    assert head == wanted, (head, wanted)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements.txt')], check=True)
    return REPO

def write_runner_config():
    payload = {name: str(value) if isinstance(value, Path) else value for name, value in {
        'DRIVE_ROOT': DRIVE_ROOT, 'DATASET_ROOT': DATASET_ROOT,
        'NNUNET_RESULTS': NNUNET_RESULTS, 'TRACKB_CACHE_ROOT': TRACKB_CACHE_ROOT,
        'OUTPUT_ROOT': OUTPUT_ROOT, 'REPO_URL': REPO_URL,
        'PINNED_COMMIT': PINNED_COMMIT, 'NUM_WORKERS': NUM_WORKERS,
        'DEVICE': DEVICE, 'MAX_CASES': MAX_CASES, 'FORCE_REBUILD': FORCE_REBUILD,
        'EXPORT_TRUE_SOFTMAX': EXPORT_TRUE_SOFTMAX, 'RETRY_FAILED': RETRY_FAILED}.items()}
    CONFIG_PATH.write_text(json.dumps(payload, indent=2))
    return CONFIG_PATH


## Cell group 1 — environment: mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO = bootstrap_repo()
import torch
assert DEVICE == 'cuda' and torch.cuda.is_available(), 'Prompt-1 OOF completion requires a CUDA runtime'
for name, path in [('DATASET_ROOT', DATASET_ROOT), ('NNUNET_RESULTS', NNUNET_RESULTS)]:
    assert path.is_dir(), f'{name} missing: {path}'
for name, path in [('TRACKB_CACHE_ROOT', TRACKB_CACHE_ROOT), ('OUTPUT_ROOT', OUTPUT_ROOT)]:
    path.mkdir(parents=True, exist_ok=True)
    assert path.resolve().is_relative_to(DRIVE_ROOT.resolve()), f'{name} must be under DRIVE_ROOT'
smoke_file = OUTPUT_ROOT / '.prompt1_drive_rw_smoke'
smoke_file.write_text('ok'); assert smoke_file.read_text() == 'ok'; smoke_file.unlink()
for path in (Path('/content'), DRIVE_ROOT):
    usage = shutil.disk_usage(path)
    print(path, {'total_GiB': round(usage.total/2**30, 2), 'free_GiB': round(usage.free/2**30, 2)})
print('CUDA:', torch.cuda.get_device_name(0))
print('Pinned SHA:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Runner config:', write_runner_config())


## Cell group 2 — legacy provenance bootstrap, read-only OOF audit, and 40-case identity preflight

This is the hard stop. For the exactly 40 currently complete legacy `oof_probs` cases, the runner first performs case-resumable CUDA re-prediction with each case's validation-fold checkpoint, requires voxel-wise equality with the untouched legacy hard mask, writes exact provenance plus official softmax, and only then runs the audit and three-path identity preflight. Any mismatch, geometry, SDF round-trip, or full-path identity failure stops the completion queue.

In [ ]:
REPO = bootstrap_repo()
cfg = write_runner_config()
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=REPO, check=True)
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'preflight'], cwd=REPO, check=True)


## Cell group 3 — full 480-case development manifest

The manifest records images, labels, expected fold, hard OOF, separate true-softmax, both SDF caches, geometry, provenance, checksums, validity and timestamps in `OUTPUT_ROOT/prompt1/cache_manifest_480.json`. It asserts that the 52 S cases are absent.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'manifest'], cwd=REPO, check=True)


## Cell group 4 — fold-aware missing OOF generation

RUN/RESUME first performs two actual single-case CUDA predictions. Each uses only its validation-fold model. Local `/content` hard and official nnU-Net softmax exports are validated, then atomically published to separate Drive directories. Existing valid hard artifacts are compared during smoke and never overwritten.

## Cell group 5 — physical-mm SDF completion

Missing coarse and GT SDFs are computed case by case with physical spacing. CPU concurrency is capped at two, errors enter the bounded retry list, and every completed cache is atomically published.

## Cell group 6 — full cache validation

The manifest is rebuilt and the identity gate requires 480 valid, zero missing, zero invalid, and zero fold-provenance errors. True-softmax coverage is reported separately and a hard mask is never substituted for it.

## Cell group 7 — full three-path identity baseline

Only after the cache gate, the runner measures direct hard, SDF-sign, and full zero-velocity paths per side, case, fold, and overall; reports H1–H4; and writes `identity_prior.json`, `identity_prior_cases.csv`, `identity_prior_folds.csv`, and `identity_prior_report.md`. `identity_baseline.py --write-config` is invoked only from a validated 480-case report. The non-inferiority margin remains null.

## Cell group 8 — overnight state and RUN/RESUME

State is atomically updated on Drive after every case, with SHA, stage, completed/failed cases, retries, checksums, heartbeat, fold/case, CUDA device and session ID. SIGTERM/KeyboardInterrupt saves state. `MAX_CASES=2` stops honestly with `complete_cv=false`; set it to `None` for the full run.

In [ ]:
# RUN/RESUME — safe to execute again after a disconnect.
REPO = bootstrap_repo()
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'run'], cwd=REPO, check=True)
